# 07 – M6: mDeBERTa đã học NLI (W4-10)

**Người chạy: Quang.** Chạy lần lượt từ trên xuống, mỗi ô chờ xong mới chạy ô tiếp. Tổng khoảng **60–75 phút** GPU.

**Bắt buộc dùng GPU:** Runtime → Change runtime type → **T4 GPU** → Save.

M6 = `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`: mDeBERTa-v3-base đã được huấn luyện NLI trên khoảng 2,7 triệu
cặp câu ở 27 ngôn ngữ (có tiếng Việt). Notebook làm 2 việc:
- **S0:** đánh giá thẳng trên 3 tập test, không huấn luyện thêm (khoảng 3 phút);
- **S2:** fine-tune thêm trên ViANLI train, so 2 cấu hình A/B trên dev, rồi chạy thêm seed 123, 2024 với cấu hình được chọn.

Kết quả tự lưu vào Drive chung `NLI/results/`. M6 dùng văn bản thô, **không cần** file tách từ.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results', 'checkpoints']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

## Kiểm tra GPU và dữ liệu

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU, bật T4 GPU rồi chạy lại từ ô đầu')
need = [f'data/processed/{f}.jsonl' for f in ['vianli_train', 'vianli_dev', 'vianli_test', 'vinli_test', 'xnli_test']]
missing = [f for f in need if not os.path.exists(f)]
print('Thiếu file dữ liệu:', missing if missing else 'không, đủ cả 5 file')

**Đạt khi:** dòng GPU ghi `Tesla T4` và không thiếu file. Thiếu file thì dừng lại, nhắn Quân.

## Bước 1: chạy thử (khoảng 3 phút)
1.000 mẫu train, 1 epoch, kết quả lưu riêng ở `results/smoke/`, không lẫn với kết quả chính.

In [ ]:
!python scripts/run_transformer.py --model M6 --scenario S2 --smoke

In [ ]:
import json, glob, math
for f in sorted(glob.glob('results/smoke/M6_S2_*.json')):
    r = json.load(open(f)); c = r['config']
    print(f"{r['test_set']}: n={r['n']}, accuracy={r['accuracy']:.3f}")
print('thiết bị:', c['device'], '| fp16:', c['fp16'], '| loss:', [h['train_loss'] for h in c['history']])
print('LOSS BỊ NAN → xem hướng dẫn bên dưới' if any(h['train_loss'] is None or math.isnan(h['train_loss']) for h in c['history'])
      else 'OK, chạy tiếp bước 2')

**Đạt khi:** không có lỗi, `thiết bị: cuda`, `fp16: True`, loss là một số (khoảng 0,5–1,5) và dòng cuối ghi `OK`.
Accuracy chạy thử thường đã khoảng 0,4–0,6 vì mô hình học NLI từ trước, không giống PhoBERT chạy thử (≈ 0,33).

**Nếu loss bị nan:** thêm ` --no-fp16` vào cuối **mọi** lệnh `run_transformer.py` bên dưới (chạy chậm hơn khoảng 2 lần) và chạy lại ô chạy thử để kiểm tra.

## Bước 2: S0 – đánh giá thẳng, không fine-tune (khoảng 3 phút)

In [ ]:
!python scripts/run_transformer.py --model M6 --scenario S0

## Bước 3: S2 – so cấu hình A và B trên dev, seed 42 (khoảng 25 phút)
Giống W3-03 của PhoBERT: mỗi cấu hình lưu riêng ở `results/tuning/`, cấu hình có macro-F1 **dev** cao hơn được chép ra `results/`.
Chỉ dùng dev để chọn, không nhìn test.

In [ ]:
!python scripts/run_transformer.py --model M6 --scenario S2 --config A B

In [ ]:
import json
t = json.load(open('results/analysis/M6_tuning_S2_seed42.json'))
for r in t['results']:
    print(f"cấu hình {r['config']}: dev macro-F1 {r['dev_macro_f1']:.3f}, tốt nhất ở epoch {r['best_epoch']}, "
          f"{round(r['train_seconds'] / 60)} phút")
BEST = t['chosen']
print('→ chọn cấu hình', BEST)

## Bước 4: thêm seed 123 và 2024 với cấu hình đã chọn (khoảng 15–30 phút)

In [ ]:
!python scripts/run_transformer.py --model M6 --scenario S2 --config {BEST} --seeds 123 2024

## Bước 5: ensemble và bảng tổng hợp (vài giây, không cần GPU)
`ensemble.py` kết hợp M6 với PhoBERT, XLM-R bằng bỏ phiếu đa số (thành viên cố định từ trước, phá hòa theo dev).

In [ ]:
!python scripts/ensemble.py
!python scripts/make_table.py
!python scripts/plot_results.py

In [ ]:
import glob
need = (['results/M6_S0_%s_seed42.json' % t for t in ['xnli_test', 'vianli_test', 'vinli_test']] +
        ['results/M6_S2_%s_seed%d.json' % (t, s) for t in ['vianli_test', 'vinli_test'] for s in [42, 123, 2024]] +
        ['results/analysis/M6_tuning_S2_seed42.json', 'results/analysis/ensemble.json'])
missing = [f for f in need if not glob.glob(f)]
print('THIẾU:', missing) if missing else print('Đủ 11 file kết quả. Xong W4-10.')

## Lưu và nộp
- Kết quả đã nằm trên Drive chung `NLI/results/` (không cần tải về). Ô trên phải in `Đủ 11 file kết quả`.
- M6 không lưu checkpoint lên Drive (S2 không cần học tiếp), nên không tốn chỗ Drive.
- Trên sheet: W4-10 → **Đang review**, ghi chú cấu hình được chọn và dòng in của ô `ensemble.py`. Nhắn Quân để đồng bộ lên GitHub.